# Validate the JSON-LD format of Manus Agent

In [3]:
!python validate_jsonld.py 1-2107.07653v3.jsonld


File: 1-2107.07653v3.jsonld
────────────────────────────────────────────────────────────

[Valid JSON]
  ✔  PASS
  File is valid JSON.

[@context present]
  ✔  PASS
  @context is present.

[@graph structure]
  ✔  PASS
  @graph is a valid array of 47 node(s).

[@id uniqueness]
  ✔  PASS
  All 47 @id values are unique.

[Character offsets]
  ⚠  WARN
  166 object(s) are missing character offsets:
      • study:tapex_table_pretraining_2022  (missing: start_char_doc, end_char_doc)
      • person:liu_qian  (missing: start_char_doc, end_char_doc)
      • person:chen_bei  (missing: start_char_doc, end_char_doc)
      • person:guo_jiaqi  (missing: start_char_doc, end_char_doc)
      • person:ziyadi_morteza  (missing: start_char_doc, end_char_doc)
      • person:lin_zeqi  (missing: start_char_doc, end_char_doc)
      • person:chen_weizhu  (missing: start_char_doc, end_char_doc)
      • person:lou_jian_guang  (missing: start_char_doc, end_char_doc)
      • org:beihang_university  (missing: start

# Flatten the GSAP Test Set into .csv file (one for entities and one for relations)

**From JSONL to CSV**

In [13]:
import json
import csv

def extract_entity_text(tokens, start_idx, end_idx):
    return ' '.join(tokens[start_idx:end_idx+1])

def flatten_ner(ner_data):
    """Convert nested NER lists into a flat list of [start, end, label]."""
    flat = []
    for item in ner_data:
        if isinstance(item, list) and len(item) == 3 and isinstance(item[2], str):
            flat.append(item)
        elif isinstance(item, list):
            for sub in item:
                if isinstance(sub, list) and len(sub) == 3:
                    flat.append(sub)
                else:
                    print(f"Unexpected nested structure: {sub}")
    return flat

def process_document(doc_data, doc_id, all_entities, all_relations, entity_counter):
    """Process one JSON document and append to all_entities/all_relations."""
    all_tokens = [token for sent in doc_data['sentences'] for token in sent]
    token_spans = doc_data['doc_token_spans']

    flat_ner = flatten_ner(doc_data['ner'])
    print(f"Doc {doc_id}: Flattened NER entries: {len(flat_ner)}")

    # Entities for this document
    doc_entities = []
    for i, ner_entry in enumerate(flat_ner):
        if len(ner_entry) < 3:
            print(f"Warning: NER entry {i} has {len(ner_entry)} values, skipping: {ner_entry}")
            continue
        start_idx, end_idx, label = ner_entry[:3]
        start_idx, end_idx = int(start_idx), int(end_idx)
        if start_idx >= len(all_tokens) or end_idx >= len(all_tokens):
            print(f"Warning: NER entry {i} has out-of-range token indices, skipping")
            continue
        text = extract_entity_text(all_tokens, start_idx, end_idx)
        start_char = token_spans[start_idx][0]
        end_char = token_spans[end_idx][1]

        global_entity_id = entity_counter
        entity_counter += 1

        entity = {
            'global_entity_id': global_entity_id,
            'doc_id': doc_id,
            'local_entity_id': i,
            'text': text,
            'start_char': start_char,
            'end_char': end_char,
            'ner_label': label,
            'start_token': start_idx,
            'end_token': end_idx,
            'relations_out': [],
            'relations_in': []
        }
        doc_entities.append(entity)
        all_entities.append(entity)

    # Relations
    flat_rels = []
    for rel_item in doc_data['relations']:
        if isinstance(rel_item, list) and len(rel_item) >= 5 and isinstance(rel_item[4], str):
            flat_rels.append(rel_item[:5])
        elif isinstance(rel_item, list):
            for sub in rel_item:
                if isinstance(sub, list) and len(sub) >= 5:
                    flat_rels.append(sub[:5])

    # Build mapping from token span to entity for this document
    span_to_entity = {}
    for ent in doc_entities:
        key = (ent['start_token'], ent['end_token'])
        span_to_entity[key] = ent

    for rel_entry in flat_rels:
        if len(rel_entry) < 5:
            print(f"Warning: Relation entry has {len(rel_entry)} values, skipping: {rel_entry}")
            continue
        h_start, h_end, t_start, t_end, rel_type = rel_entry[:5]
        h_start, h_end, t_start, t_end = map(int, [h_start, h_end, t_start, t_end])
        head = span_to_entity.get((h_start, h_end))
        tail = span_to_entity.get((t_start, t_end))
        if head and tail:
            head['relations_out'].append(f"{rel_type} -> {tail['text']}")
            tail['relations_in'].append(f"{rel_type} <- {head['text']}")
            all_relations.append({
                'doc_id': doc_id,
                'head_entity_id': head['global_entity_id'],
                'head_text': head['text'],
                'relation_type': rel_type,
                'tail_entity_id': tail['global_entity_id'],
                'tail_text': tail['text']
            })
        else:
            print(f"Warning: Could not find entities for relation {rel_entry}")

    return entity_counter

def convert_jsonl_to_csv(jsonl_path, entities_csv_path, relations_csv_path):
    all_entities = []
    all_relations = []
    entity_counter = 0

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                doc_data = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"Error parsing line {line_num}: {e}")
                continue

            # Get document ID (use line number if missing)
            doc_id = doc_data.get('doc_id', f'doc_{line_num}')
            entity_counter = process_document(doc_data, doc_id, all_entities, all_relations, entity_counter)

    print(f"\nTotal entities across all documents: {len(all_entities)}")
    print(f"Total relations across all documents: {len(all_relations)}")

    # Write entities CSV
    with open(entities_csv_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['global_entity_id', 'doc_id', 'local_entity_id', 'text',
                      'start_char', 'end_char', 'ner_label', 'relations_out', 'relations_in']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for ent in all_entities:
            writer.writerow({
                'global_entity_id': ent['global_entity_id'],
                'doc_id': ent['doc_id'],
                'local_entity_id': ent['local_entity_id'],
                'text': ent['text'],
                'start_char': ent['start_char'],
                'end_char': ent['end_char'],
                'ner_label': ent['ner_label'],
                'relations_out': '; '.join(ent['relations_out']),
                'relations_in': '; '.join(ent['relations_in'])
            })

    # Write relations CSV
    with open(relations_csv_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['doc_id', 'head_entity_id', 'head_text', 'relation_type',
                      'tail_entity_id', 'tail_text']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for rel in all_relations:
            writer.writerow(rel)

    print(f"Entities CSV saved to: {entities_csv_path}")
    print(f"Relations CSV saved to: {relations_csv_path}")

# Usage
convert_jsonl_to_csv('xxx.jsonl', 'entities.csv', 'relations.csv')

Doc 00001_1911_02116.txt: Flattened NER entries: 772
Doc 00013_2104_08727.txt: Flattened NER entries: 638
Doc 00030_2107_07653.txt: Flattened NER entries: 716
Doc 00040_1908_08962.txt: Flattened NER entries: 739
Doc 00050_1406_2661.txt: Flattened NER entries: 333
Doc 10018_2208_04405.txt: Flattened NER entries: 255
Doc 10019_2204_02965.txt: Flattened NER entries: 480
Doc 10037_1804_05260.txt: Flattened NER entries: 1191
Doc 10040_2112_07839.txt: Flattened NER entries: 714
Doc 10047_2005_06182.txt: Flattened NER entries: 300

Total entities across all documents: 6138
Total relations across all documents: 3675
Entities CSV saved to: entities.csv
Relations CSV saved to: relations.csv


**Check start/end chars over the official document**

In [41]:
import csv
import os

def verify_entity_from_txt(txt_file_path, local_entity_id, csv_path, doc_id=None):
    """
    Verify an entity annotation by reading the raw .txt file directly.
    Matches doc_id as stored in CSV (including .txt extension).
    """
    # If doc_id not provided, infer from txt file name (including extension)
    if doc_id is None:
        doc_id = os.path.basename(txt_file_path)   # e.g., '00001_1911_02116.txt'

    # Read the CSV to find the entity row
    entity_row = None
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Match by doc_id and local_entity_id (both as strings)
            if row.get('doc_id') == doc_id and row.get('local_entity_id') == str(local_entity_id):
                entity_row = row
                break

    if entity_row is None:
        print(f"Entity not found: doc_id='{doc_id}', local_entity_id={local_entity_id}")
        return

    # Extract fields
    start = int(entity_row['start_char'])
    end = int(entity_row['end_char'])
    expected_text = entity_row['text']
    ner_label = entity_row['ner_label']

    # Read raw text from .txt file
    with open(txt_file_path, 'r', encoding='utf-8') as f:
        raw_text = f.read()

    extracted = raw_text[start:end]

    print(f"Document: {doc_id}")
    print(f"Local Entity ID: {local_entity_id} ({ner_label})")
    print(f"Extracted: '{extracted}'")
    print(f"Expected:  '{expected_text}'")
    print(f"Match: {extracted == expected_text}")

    # Show context
    context_start = max(0, start - 30)
    context_end = min(len(raw_text), end + 30)
    print(f"Context: ...{raw_text[context_start:context_end]}...\n")

# Example usage
verify_entity_from_txt(
    txt_file_path='/content/10040_2112_07839.txt',
    local_entity_id=2,
    csv_path='/content/All NER Entities GSAP Test Set.csv'
)

Document: 10040_2112_07839.txt
Local Entity ID: 2 (MLModelGeneric)
Extracted: 'a learning model'
Expected:  'a learning model'
Match: True
Context: ...s at collaboratively training a learning model across a large number of dist...



**Convert document raw text .json into .txt file**

In [27]:
import json

# Load JSON with the "text" field
with open('/content/10047_2005_06182.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

raw_text = data['text']

# Write exactly as is (no extra newline at end unless present in original)
with open('/content/10047_2005_06182.txt', 'w', encoding='utf-8', newline='') as f:
    f.write(raw_text)